# 모의면접 질문생성 모델 학습 (Job)

AI Hub "채용면접 인터뷰 데이터"로 한국어 언어모델(KoGPT2)을 파인튜닝해서, 직무/이전 답변을 넣으면 다음 면접 질문을 생성하는 모델을 만든다.

**전체 단계**
1. 런타임을 GPU로 설정
2. 라이브러리 설치
3. AI Hub 데이터 업로드 & 압축 해제
4. 데이터 구조 확인 (여기서 잠깐 멈춤 - 실제 구조 보고 다음 셀을 같이 채운다)
5. 질문 텍스트만 뽑아서 학습용 포맷으로 변환
6. KoGPT2 모델/토크나이저 로드
7. LoRA 설정 (전체 파인튜닝 대신 가벼운 부분만 학습 - 무료 GPU로도 충분)
8. 학습
9. 테스트 (질문 생성해보기)
10. 모델 저장

**0단계 (지금 바로 할 것)**: 코랩 메뉴에서 `런타임 > 런타임 유형 변경 > 하드웨어 가속기 > T4 GPU` 선택하고 저장. 이거 안 하면 학습이 엄청 느려.

## 1. 라이브러리 설치
`transformers`: 모델/토크나이저, `peft`: LoRA(가벼운 파인튜닝), `accelerate`: 학습 속도 최적화, `datasets`: 데이터 다루기 편하게.

In [ ]:
!pip install -q transformers peft accelerate datasets bitsandbytes

import torch
print("GPU 사용 가능:", torch.cuda.is_available())
print("GPU 이름:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음 (런타임을 GPU로 바꿔줘)")

## 2. 데이터 업로드

AI Hub에서 받은 zip 파일을 구글 드라이브에 올려두고 마운트하는 방식을 추천 (코랩에 직접 업로드하면 세션 끊길 때마다 다시 올려야 함).

1. 구글 드라이브에 `MyDrive/jobara_ml/` 폴더 만들고 다운받은 zip 파일을 거기 업로드
2. 아래 셀 실행하면 드라이브 접근 권한 요청 팝업 뜸 - 허용

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# TODO: 실제 업로드한 zip 파일 이름으로 바꿔줘
ZIP_PATH = "/content/drive/MyDrive/jobara_ml/채용면접_인터뷰_데이터.zip"
EXTRACT_DIR = "/content/interview_data"

import zipfile, os
os.makedirs(EXTRACT_DIR, exist_ok=True)
with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
    zf.extractall(EXTRACT_DIR)

print("압축 해제 완료")

## 3. 데이터 구조 확인 (중요 - 여기서 멈추고 결과를 같이 봐야 함)

AI Hub 데이터는 데이터셋마다 폴더/JSON 구조가 다 달라서, 실제로 열어보기 전엔 다음 단계(질문 텍스트 추출)를 정확히 못 짠다.
아래 셀 실행하고 **출력 결과를 캡처해서 보여주면**, 그걸 보고 5단계 파싱 코드를 정확하게 채워줄게.

In [ ]:
import os

# 폴더 구조를 3단계 깊이까지만 출력 (너무 길면 위쪽만 캡처해서 보여줘도 됨)
def print_tree(path, prefix="", depth=0, max_depth=3):
    if depth > max_depth:
        return
    try:
        entries = sorted(os.listdir(path))
    except NotADirectoryError:
        return
    for entry in entries[:15]:  # 폴더당 15개까지만
        full = os.path.join(path, entry)
        print(prefix + entry)
        if os.path.isdir(full):
            print_tree(full, prefix + "  ", depth + 1, max_depth)

print_tree(EXTRACT_DIR)

In [ ]:
# 위에서 찾은 JSON 라벨 파일 하나를 직접 열어서 내용 구조를 본다.
# TODO: 위 출력에서 실제 .json 파일 경로 하나를 찾아서 아래에 넣어줘
import json

SAMPLE_JSON_PATH = "여기에_실제_json_파일_경로"  # 예: /content/interview_data/Training/label/xxx.json

with open(SAMPLE_JSON_PATH, encoding="utf-8") as f:
    sample = json.load(f)

print(json.dumps(sample, ensure_ascii=False, indent=2)[:2000])

## 4. 질문 텍스트 추출 (3단계 결과 보고 같이 채우는 부분)

지금은 자리만 잡아둔 상태 - 실제 JSON 키 이름(예: `question`, `qa`, `utterance` 등)을 알아야 정확히 짤 수 있다.

In [ ]:
# TODO(3단계 결과 확인 후 채움): 모든 JSON을 순회하며 (직무, 이전맥락, 질문) 튜플 리스트를 만든다.
# questions = []
# for root, _, files in os.walk(EXTRACT_DIR):
#     for fname in files:
#         if not fname.endswith(".json"):
#             continue
#         with open(os.path.join(root, fname), encoding="utf-8") as f:
#             data = json.load(f)
#         # 실제 키 구조에 맞춰 아래를 채운다
#         # questions.append({"job": ..., "context": ..., "question": ...})
#
# print(f"{len(questions)}개 질문 수집")
raise NotImplementedError("3단계 출력 보여주면 이 셀 채워줄게")

## 5. 모델/토크나이저 로드
SKT KoGPT2(1.2억 파라미터) - 가볍고 한국어 생성에 무난한 베이스 모델.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "skt/kogpt2-base-v2"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME, bos_token="</s>", eos_token="</s>", unk_token="<unk>", pad_token="<pad>"
)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
print("모델 파라미터 수:", sum(p.numel() for p in model.parameters()) / 1e6, "M")

## 6. LoRA 설정

모델 전체(1.2억 파라미터)를 다 학습시키는 대신, 일부 레이어에 작은 "어댑터"만 붙여서 그것만 학습한다.

- 학습 속도 훨씬 빠름, GPU 메모리 훨씬 적게 씀 (무료 T4로 충분)
- `r`: 어댑터 크기 (작을수록 가볍지만 표현력 낮음, 8~16이 보통 무난)

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["c_attn"],  # GPT2 계열 어텐션 레이어
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()  # 전체 대비 학습되는 파라미터 비율 확인 (보통 1% 미만)

## 7. 학습 데이터셋 구성

4단계에서 만든 `questions` 리스트를 "직무: OO\n이전 답변: ...\n다음 질문: 실제질문" 형태의 텍스트로 바꿔서 토크나이징한다.
모델은 이 패턴을 배워서, "다음 질문:" 뒤를 이어 쓰는 방식으로 새 질문을 생성하게 된다.

In [ ]:
from datasets import Dataset

def to_prompt(item):
    job = item.get("job", "IT")
    context = item.get("context", "")
    question = item["question"]
    return f"직무: {job}\n이전 답변: {context}\n다음 질문: {question}{tokenizer.eos_token}"

texts = [to_prompt(q) for q in questions]  # 4단계에서 만든 questions 사용

def tokenize_fn(batch):
    out = tokenizer(batch["text"], truncation=True, max_length=256, padding="max_length")
    out["labels"] = out["input_ids"].copy()
    return out

raw_ds = Dataset.from_dict({"text": texts})
tokenized_ds = raw_ds.map(tokenize_fn, batched=True, remove_columns=["text"])
print(tokenized_ds)

## 8. 학습
`num_train_epochs`, `per_device_train_batch_size`는 데이터 양 보고 나중에 같이 조정하자 (데이터 몇 개 나오는지에 따라 다름).

In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="/content/checkpoints",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=20,
    save_strategy="epoch",
    report_to="none",
)

trainer = Trainer(model=model, args=training_args, train_dataset=tokenized_ds)
trainer.train()

## 9. 테스트 - 질문 생성해보기

In [ ]:
def generate_question(job: str, context: str = "") -> str:
    prompt = f"직무: {job}\n이전 답변: {context}\n다음 질문:"
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    output = model.generate(
        **inputs, max_new_tokens=40, do_sample=True, top_p=0.9, temperature=0.8,
        pad_token_id=tokenizer.pad_token_id,
    )
    text = tokenizer.decode(output[0], skip_special_tokens=True)
    return text.split("다음 질문:")[-1].strip()

print(generate_question("백엔드 개발자"))
print(generate_question("프론트엔드 개발자", context="React로 SPA를 개발한 경험이 있습니다."))

## 10. 모델 저장
LoRA 어댑터만 저장하면 되니까 용량 작음 (몇십 MB 수준). 나중에 우리 ai-server(FastAPI)에서 이 어댑터 파일만 불러와서 쓰면 된다.

In [ ]:
SAVE_PATH = "/content/drive/MyDrive/jobara_ml/question_generator_lora"
model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)
print("저장 완료:", SAVE_PATH)